In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM MBPAE10", con=connection2)

connection2.close()



In [2]:
base2

,priatecod,priatedes,priatedescor,priatetriflg
0,0,CADAVER (SIN PRIORIDAD),S.P.,1
1,1,PRIORIDAD I - RIESGO VIDA/RESUCITAC,P.I,1
2,2,PRIORIDAD II - EMERGENCIA,P.II,1
3,3,PRIORIDAD III - URGENCIA MAYOR,P.III,1
4,4,PRIORIDAD IV - URGENCIA MENOR,P.IV,1
5,5,PRIORIDAD V - CONSULTA,P.V,1


In [3]:
a=base2[base2.duplicated(subset=["priatecod"])]
a

,priatecod,priatedes,priatedescor,priatetriflg


In [4]:
base2.columns

Index(['priatecod', 'priatedes', 'priatedescor', 'priatetriflg'], dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



base2.to_sql(name='tmp_mbpae10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL mbpae10 FALSO CON LO OBTENIDO DEL ORACLE

query_count_before = "SELECT COUNT(*) FROM mbpae10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla mbpae10 antes de la inserción: {cant_antes}")



query="""
ALTER TABLE tmp_mbpae10 
ALTER COLUMN priatecod TYPE CHARACTER (1),
ALTER COLUMN priatedes TYPE CHARACTER (35),
ALTER COLUMN priatedescor TYPE CHARACTER (5),
ALTER COLUMN priatetriflg TYPE CHARACTER (1);


UPDATE mbpae10 
SET 

priatedes= t.priatedes,
priatedescor= t.priatedescor,
priatetriflg= t.priatetriflg


FROM tmp_mbpae10 t 
WHERE mbpae10.priatecod = t.priatecod and mbpae10.priatecod IS NOT NULL and t.priatecod IS NOT NULL ;

INSERT INTO mbpae10 
(priatecod, priatedes, priatedescor, priatetriflg) 

SELECT 
priatecod, priatedes, priatedescor, priatetriflg

FROM tmp_mbpae10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM mbpae10 
    WHERE mbpae10.priatecod = tmp_mbpae10.priatecod and mbpae10.priatecod IS NOT NULL and tmp_mbpae10.priatecod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_mbpae10;
DELETE FROM mbpae10 WHERE priatecod ='';
SELECT COUNT(*) FROM mbpae10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla mbpae10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

base2 = pd.read_sql_query(f"SELECT * FROM mbpae10", con=connection3)

connection3.close()


Cantidad de filas en la tabla mbpae10 antes de la inserción: 6
Cantidad de filas en la tabla mbpae10 después de la inserción: 6
La cantidad de filas insertadas fue de: 0


In [6]:
base2.columns

Index(['priatecod', 'priatedes', 'priatedescor', 'priatetriflg'], dtype='object')

In [7]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_mbpae10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query_count_before = "SELECT COUNT(*) FROM dim_emepri"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla dim_emepri antes de la inserción: {cant_antes}")


query="""

ALTER TABLE tmp_mbpae10 
ALTER COLUMN priatecod TYPE CHARACTER (1),
ALTER COLUMN priatedes TYPE CHARACTER (35),
ALTER COLUMN priatedescor TYPE CHARACTER (5),
ALTER COLUMN priatetriflg TYPE CHARACTER (1);



INSERT INTO dim_emepri (cod_pri, des_pri,abr_pri,fla_pri) 
SELECT priatecod, priatedes,priatedescor,priatetriflg

FROM tmp_mbpae10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_emepri 
    WHERE dim_emepri.cod_pri = tmp_mbpae10.priatecod
);

DROP TABLE tmp_mbpae10;

SELECT COUNT(*) FROM dim_emepri;
"""

c1 = text(query)
cursor = connection4.execute(c1)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla dim_emepri después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

connection4.close()

Cantidad de filas en la tabla dim_emepri antes de la inserción: 6
Cantidad de filas en la tabla dim_emepri después de la inserción: 6
La cantidad de filas insertadas fue de: 0
